# Conditional Diffusion for Image Colorization

In this notebook, we build a **conditional latent diffusion model** that colorizes grayscale images using a color reference image. The reference image's color palette is injected via **cross-attention** — the same mechanism Stable Diffusion uses for CLIP text embeddings.

### Learning Objectives
- Refresh the mathematical foundations of diffusion models (forward & reverse processes)
- Understand how to **condition** diffusion models on external signals (images, text)
- Learn the two main conditioning mechanisms: **channel concatenation** and **cross-attention**
- See how our approach mirrors the Stable Diffusion architecture
- Train a working colorization model on COCO images (~70 min on an L4 GPU)

### Prerequisites
You should be familiar with:
- DDPM (Denoising Diffusion Probabilistic Models)
- VAEs (Variational Autoencoders) — encoder/decoder, latent space
- Attention mechanisms and Transformers
- Latent Diffusion Models (LDMs) — the idea of running diffusion in latent space

---
## 1. Diffusion Refresher: Forward Process

A diffusion model defines a **forward (noising) process** that gradually corrupts data into pure Gaussian noise, and a **reverse (denoising) process** that learns to undo the corruption.

### Forward Process

Given a clean data sample $x_0$, we define a Markov chain that adds Gaussian noise over $T$ timesteps:

$$q(x_t \mid x_{t-1}) = \mathcal{N}\left(x_t;\; \sqrt{1 - \beta_t}\, x_{t-1},\; \beta_t \mathbf{I}\right)$$

where $\beta_t \in (0, 1)$ is the **noise schedule**. We use a linear schedule: $\beta_1 = 10^{-4}$ to $\beta_T = 0.02$, with $T = 1000$.

### Closed-Form Sampling

A key insight is that we can sample $x_t$ directly from $x_0$ without iterating through all intermediate steps. Define:

$$\alpha_t = 1 - \beta_t, \qquad \bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$$

Then:

$$q(x_t \mid x_0) = \mathcal{N}\left(x_t;\; \sqrt{\bar{\alpha}_t}\, x_0,\; (1 - \bar{\alpha}_t) \mathbf{I}\right)$$

Using the **reparameterization trick**:

$$\boxed{x_t = \sqrt{\bar{\alpha}_t}\, x_0 + \sqrt{1 - \bar{\alpha}_t}\, \varepsilon}, \quad \varepsilon \sim \mathcal{N}(0, \mathbf{I})$$

**Intuition**: As $t$ increases, $\bar{\alpha}_t \to 0$, so $x_t$ becomes pure noise. The signal from $x_0$ is gradually lost.

## 2. Diffusion Refresher: Reverse Process & Training

### Reverse Process

The reverse process learns to denoise, stepping from $x_t$ back to $x_{t-1}$:

$$p_\theta(x_{t-1} \mid x_t) = \mathcal{N}\left(x_{t-1};\; \mu_\theta(x_t, t),\; \sigma_t^2 \mathbf{I}\right)$$

A neural network $\varepsilon_\theta$ parameterizes this by predicting the noise $\varepsilon$ that was added at timestep $t$.

### Simplified Training Objective ($\varepsilon$-prediction)

Instead of predicting $\mu_\theta$ directly, we train the network to predict the noise:

$$\boxed{\mathcal{L} = \mathbb{E}_{x_0,\, \varepsilon \sim \mathcal{N}(0,\mathbf{I}),\, t \sim \text{Uniform}(1,T)} \left[\| \varepsilon - \varepsilon_\theta(x_t, t) \|^2 \right]}$$

This is a simple MSE loss between the actual noise and the predicted noise.

### DDPM vs DDIM Sampling

**DDPM** (Denoising Diffusion Probabilistic Models): stochastic sampling, requires all $T = 1000$ steps. Slow but high quality.

**DDIM** (Denoising Diffusion Implicit Models): deterministic sampling with the **same trained model**, but allows far fewer steps (e.g., 50). The DDIM update rule:

$$\hat{x}_0 = \frac{x_t - \sqrt{1 - \bar{\alpha}_t}\, \varepsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}$$

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}}\, \hat{x}_0 + \sqrt{1 - \bar{\alpha}_{t-1}}\, \varepsilon_\theta(x_t, t)$$

**Key insight**: We train with the DDPM loss, but sample with DDIM for 20× faster inference.

---
## 3. Conditioning Diffusion Models

So far, diffusion models generate samples unconditionally: $p_\theta(x_{t-1} \mid x_t)$. To control generation, we condition on an external signal $c$:

$$p_\theta(x_{t-1} \mid x_t, c)$$

How do we inject $c$ into the network? There are two main mechanisms:

### Mechanism 1: Channel Concatenation

Append the condition directly to the noisy input along the channel dimension:

$$\text{input} = [x_t \;\|\; c_{\text{spatial}}] \in \mathbb{R}^{B \times (C + C') \times H \times W}$$

- **Pros**: Simple, preserves spatial alignment
- **Cons**: Condition must have the same spatial resolution as $x_t$
- **Best for**: Spatially-aligned signals (e.g., a grayscale image guiding colorization)

### Mechanism 2: Cross-Attention

Encode the condition into a sequence of tokens, then inject via cross-attention layers inside the U-Net:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

where:
- $Q = W_Q \cdot z_{\text{features}}$ — queries from U-Net intermediate features
- $K = W_K \cdot c_{\text{tokens}}$ — keys from the condition encoding
- $V = W_V \cdot c_{\text{tokens}}$ — values from the condition encoding

- **Pros**: Flexible, works with any encoded signal (text, images, audio)
- **Cons**: More parameters, condition doesn't need spatial alignment
- **Best for**: Semantic/global conditions (e.g., text prompts, style references)

### Our Design: Both!

For our colorization task, we combine both mechanisms:

| Condition | Mechanism | Rationale |
|-----------|-----------|----------|
| Grayscale image | Channel concatenation | Spatial alignment matters — the grayscale provides the structure |
| Reference color image | Cross-attention | Global color palette transfer — we want color information, not pixel alignment |

## 4. VAE & Latent Space Refresher

Running diffusion directly on $64 \times 64 \times 3$ images is expensive. **Latent Diffusion Models (LDMs)** first compress images into a low-dimensional latent space using a VAE, then run diffusion there.

### The VAE Pipeline

$$x \in \mathbb{R}^{64 \times 64 \times 3} \xrightarrow{\;\mathcal{E}\;} z \in \mathbb{R}^{8 \times 8 \times 4} \xrightarrow{\;\text{diffusion}\;} \hat{z} \in \mathbb{R}^{8 \times 8 \times 4} \xrightarrow{\;\mathcal{D}\;} \hat{x} \in \mathbb{R}^{64 \times 64 \times 3}$$

- **Encoder** $\mathcal{E}$: compresses the image by a factor of $8\times$ spatially (64 → 8) into 4 latent channels
- **Decoder** $\mathcal{D}$: reconstructs the image from latents
- **Dimensionality reduction**: $64 \times 64 \times 3 = 12{,}288$ → $8 \times 8 \times 4 = 256$ — that's **48× fewer dimensions!**

### Our VAE

We use the pre-trained Stable Diffusion VAE: [`stabilityai/sd-vae-ft-mse`](https://huggingface.co/stabilityai/sd-vae-ft-mse). It's frozen — we don't train it.

**Important detail**: The SD VAE uses a **scaling factor** of `0.18215`. After encoding, latents are multiplied by this factor to ensure they have approximately unit variance. Before decoding, we must divide by the same factor. Forgetting this produces washed-out or blown-out images.

---
## 5. Stable Diffusion Architecture & Our Approach

### Stable Diffusion

Stable Diffusion is an LDM with three components:
1. **VAE**: compresses images ↔ latents
2. **U-Net**: denoises latents, conditioned via cross-attention
3. **CLIP text encoder**: converts text prompt → sequence of 77 tokens × 768 dimensions

The CLIP tokens enter the U-Net as `encoder_hidden_states` and interact with U-Net features via cross-attention at multiple resolution levels.

### Our Simplified Version

We replace the CLIP text encoder with a **ResNet-18 image encoder** — same mechanism, different modality:

| | Stable Diffusion | Our Colorization Model |
|---|---|---|
| **Condition** | Text prompt | Reference color image |
| **Encoder** | CLIP text encoder (frozen) | ResNet-18 layer3 (frozen) + projection |
| **Token shape** | (B, 77, 768) | (B, 16, 256) |
| **Injection** | Cross-attention via `encoder_hidden_states` | Cross-attention via `encoder_hidden_states` |
| **Additional input** | — | Grayscale latent via channel concatenation |

### Full Pipeline

```
Reference Image ──► Frozen ResNet-18 (layer3) ──► Trainable Projection ──► (B, 16, 256) context tokens
                                                                                │
Grayscale ──► Frozen VAE Encoder ──► z_gray (4, 8, 8) ──────────┐            │ cross-attention
                                                                    │ concat     │ (inside U-Net)
Color Target ──► Frozen VAE Encoder ──► z_color ──► add noise ──► [x_t; z_gray] (8, 8, 8)
                                                                    │            │
                                                        UNet2DConditionModel ◄───┘
                                                                    │
                                                            ε̂ (predicted noise)
                                                                    │
                                                        DDIM sampling → VAE Decode → Colorized Image
```

The U-Net receives:
- **Input channels (8)**: 4 from the noisy color latent + 4 from the grayscale latent (concatenated)
- **Timestep $t$**: sinusoidal embedding, added inside each ResBlock
- **`encoder_hidden_states` (B, 16, 256)**: reference image tokens, injected via cross-attention layers

It outputs **4 channels**: the predicted noise $\hat{\varepsilon}$ in latent space.

---
## 6. Setup

Let's install dependencies and configure our environment. We use `diffusers` for the pre-trained VAE and the U-Net architecture, and `torchvision` for the CIFAR-10 dataset.

In [ ]:
!pip install -q diffusers accelerate transformers lpips opencv-python-headless

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.cuda.amp import autocast, GradScaler

from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, DDIMScheduler
from torchvision import transforms, models
from torchvision.datasets import CIFAR10

import contextlib
import lpips
from PIL import Image
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")

---
## 7. CIFAR-10 Data Loading

We use [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) for its 10 clean classes and zero download friction — `torchvision` handles it automatically in a single line. Native images are 32×32 and are upscaled to **64×64** using bicubic interpolation.

We select only the **6 animal classes** (bird, cat, deer, dog, frog, horse) — matching the spirit of the original COCO setup which also used animal categories. Non-animal classes (airplane, automobile, ship, truck) tend to be grey/monochrome and would reduce colorization quality.

### Pairing Strategy

For each training sample, we create a triplet:
- **Color target**: the original color image (this is what we want to reconstruct)
- **Grayscale input**: the desaturated version of the target (luminance channel replicated to 3 channels)
- **Reference image**: a *different* random image from the **same CIFAR-10 class**

By using a different image as the reference (not the target itself), the model is forced to learn **color transfer via cross-attention** — it can't simply copy pixels. It must extract the general color palette from the reference and apply it guided by the grayscale structure.

In [ ]:
# ============================================================
# Download CIFAR-10 and build per-class index (animals only)
# ============================================================
DATA_DIR = "/content/cifar10"

# Download train split (50,000 images across 10 classes, 5,000 per class)
cifar_train = CIFAR10(root=DATA_DIR, train=True, download=True)

# CIFAR-10 animal classes: bird=2, cat=3, deer=4, dog=5, frog=6, horse=7
# We skip non-animals (airplane=0, automobile=1, ship=8, truck=9) because
# vehicles tend to be grey/monochrome and hurt colorization training.
ANIMAL_CLASS_INDICES = {2, 3, 4, 5, 6, 7}  # 6 classes x 5,000 = 30,000 images

# Build a dict: {class_idx (int): [PIL Image, ...]}
# cifar_train.data is a numpy array (N, 32, 32, 3); .targets is a list of ints
category_images = {}
for img_array, label in zip(cifar_train.data, cifar_train.targets):
    if label not in ANIMAL_CLASS_INDICES:
        continue
    pil_img = Image.fromarray(img_array)
    category_images.setdefault(label, []).append(pil_img)

class_names = cifar_train.classes   # ['airplane', 'automobile', 'bird', ...]
print(f"Animal classes selected:")
for idx in sorted(ANIMAL_CLASS_INDICES):
    print(f"  {class_names[idx]} (class {idx}): {len(category_images[idx])} images")
print(f"Total images: {sum(len(v) for v in category_images.values())}")

In [ ]:
class CIFARColorizationDataset(Dataset):
    """Dataset that returns (color, grayscale, reference) triplets from CIFAR-10 images.

    - color:     original color image upscaled to 64x64 (the target)
    - grayscale: luminance of the target, replicated to 3 channels (VAE expects 3ch)
    - reference: a different image from the same CIFAR-10 class (color palette source)

    Images are stored as PIL Images in memory (CIFAR-10 is small enough).
    """

    def __init__(self, category_images: dict, image_size: int = 64):
        self.image_size = image_size
        self.transform = transforms.Compose([
            transforms.Resize(image_size, interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(image_size),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # -> [-1, 1]
        ])

        # Build flat list + per-class index groups
        self.samples = []        # (PIL Image, class_list_idx)
        self.class_indices = []  # list of lists: indices into self.samples per class

        idx = 0
        for class_idx in sorted(category_images.keys()):
            pil_images = category_images[class_idx]
            group = []
            for pil_img in pil_images:
                self.samples.append((pil_img, len(self.class_indices)))
                group.append(idx)
                idx += 1
            self.class_indices.append(group)

    def __len__(self):
        return len(self.samples)

    def _to_grayscale_3ch(self, img_tensor):
        """Convert RGB tensor [-1,1] to grayscale using luminance, keep 3 channels."""
        rgb = (img_tensor + 1) / 2                                          # [0, 1]
        gray = 0.299 * rgb[0] + 0.587 * rgb[1] + 0.114 * rgb[2]           # (H, W)
        gray_3ch = gray.unsqueeze(0).repeat(3, 1, 1)                        # (3, H, W)
        return gray_3ch * 2 - 1                                             # back to [-1, 1]

    def __getitem__(self, idx):
        pil_img, class_idx = self.samples[idx]
        color = self.transform(pil_img)
        gray = self._to_grayscale_3ch(color)

        # Pick a random *different* image from the same class as the reference
        class_group = self.class_indices[class_idx]
        ref_idx = idx
        while ref_idx == idx and len(class_group) > 1:
            ref_idx = random.choice(class_group)
        ref_pil, _ = self.samples[ref_idx]
        ref = self.transform(ref_pil)

        return {"color": color, "gray": gray, "ref": ref}

In [ ]:
# --- Create dataset and dataloader ---
dataset = CIFARColorizationDataset(category_images, image_size=64)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
print(f"Dataset size: {len(dataset)} images")
print(f"Batches per epoch: {len(dataloader)}")

# --- Visualize sample triplets ---
def show_triplets(batch, n=5):
    """Display (grayscale, reference, color target) triplets."""
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    titles = ["Grayscale (input)", "Reference (condition)", "Color (target)"]
    for i in range(n):
        for j, key in enumerate(["gray", "ref", "color"]):
            img = batch[key][i].permute(1, 2, 0).numpy() * 0.5 + 0.5  # [-1,1] -> [0,1]
            axes[i, j].imshow(img.clip(0, 1))
            axes[i, j].set_title(titles[j] if i == 0 else "")
            axes[i, j].axis("off")
    plt.tight_layout()
    plt.show()

sample_batch = next(iter(dataloader))
show_triplets(sample_batch, n=5)

---
## 8. Model Definition

Our model has four components:

| Component | Role | Source | Trainable? |
|-----------|------|--------|------------|
| **VAE** | Compress images ↔ latents (8×8×4) | `diffusers.AutoencoderKL` (frozen) | No |
| **Reference Encoder** | Encode reference image → 16 tokens × 256 dim | ResNet-18 backbone (frozen) + linear projection | Projection only (~65K params) |
| **U-Net** | Denoise latents conditioned on grayscale + reference | `diffusers.UNet2DConditionModel` | Yes (~28M params) |
| **Noise Schedulers** | Manage the noise schedule for training (DDPM) and inference (DDIM) | `diffusers.DDPMScheduler` / `DDIMScheduler` | N/A |

### How the U-Net uses conditioning

The `UNet2DConditionModel` from `diffusers` accepts an `encoder_hidden_states` argument — a tensor of shape `(B, seq_len, dim)`. Inside the U-Net, at each `CrossAttnDownBlock2D` / `CrossAttnUpBlock2D`:

1. The spatial features $z \in \mathbb{R}^{B \times C \times H \times W}$ are reshaped to a sequence: $(B, H{\cdot}W, C)$
2. **Self-attention**: $Q = K = V = z_{\text{seq}}$ — the features attend to themselves
3. **Cross-attention**: $Q = z_{\text{seq}}$, $K = V = \texttt{encoder\_hidden\_states}$ — features attend to the condition
4. **Feed-forward**: MLP applied to each token

This is identical to how CLIP text tokens condition Stable Diffusion — we just provide image tokens instead.

In [ ]:
# ============================================================
# VAE: Pre-trained Stable Diffusion VAE (frozen)
# ============================================================
vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device)
vae.eval()
for p in vae.parameters():
    p.requires_grad = False

SCALING_FACTOR = vae.config.scaling_factor  # 0.18215
print(f"VAE scaling factor: {SCALING_FACTOR}")
print(f"VAE latent channels: {vae.config.latent_channels}")  # 4


def encode_to_latent(images):
    """Encode images to VAE latent space.
    Args: images (B, 3, 64, 64) in [-1, 1]
    Returns: latents (B, 4, 8, 8)
    """
    with torch.no_grad():
        latent_dist = vae.encode(images).latent_dist
        latents = latent_dist.sample() * SCALING_FACTOR
    return latents


def decode_from_latent(latents):
    """Decode latents back to image space.
    Args: latents (B, 4, 8, 8)
    Returns: images (B, 3, 64, 64) in [-1, 1]
    """
    with torch.no_grad():
        images = vae.decode(latents / SCALING_FACTOR).sample
    return images.clamp(-1, 1)

In [ ]:
# ============================================================
# VAE Sanity Check: encode -> decode and compare
# ============================================================
test_imgs = sample_batch["color"][:4].to(device)
test_latents = encode_to_latent(test_imgs)
test_recon = decode_from_latent(test_latents)

print(f"Input shape:   {test_imgs.shape}")    # (4, 3, 64, 64)
print(f"Latent shape:  {test_latents.shape}")  # (4, 4, 8, 8)
print(f"Recon shape:   {test_recon.shape}")    # (4, 3, 64, 64)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i in range(4):
    orig = test_imgs[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
    recon = test_recon[i].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[0, i].imshow(orig.clip(0, 1))
    axes[0, i].set_title("Original" if i == 0 else "")
    axes[0, i].axis("off")
    axes[1, i].imshow(recon.clip(0, 1))
    axes[1, i].set_title("VAE Reconstruction" if i == 0 else "")
    axes[1, i].axis("off")
plt.suptitle("VAE Encode → Decode (should be near-perfect, 64×64 images)")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Reference Encoder (hand-written — the novel conditioning part)
# ============================================================

class ReferenceEncoder(nn.Module):
    """Encodes a reference color image into a sequence of tokens for cross-attention.

    This is analogous to the CLIP text encoder in Stable Diffusion:
        CLIP:  text   → 77 tokens × 768 dim
        Ours:  image  → 16 tokens × 256 dim  (for 64×64 input)

    Architecture:
        1. Frozen ResNet-18 backbone up to layer3 → (B, 256, 4, 4) for 64×64 input
        2. Reshape spatial features to token sequence → (B, 16, 256)
        3. Trainable linear projection → (B, 16, context_dim)

    For 64×64 input, ResNet-18 layer3 outputs (B, 256, 4, 4):
        - Input  64×64 → conv1+maxpool → 16×16 → layer1 → 16×16
        - layer2 → 8×8 → layer3 → 4×4
    Each of the 16 tokens corresponds to a 16×16 spatial region of the reference image.

    Efficiency design — split into two paths so backbone can be pre-computed once:
        backbone_features()  : frozen, @torch.no_grad(), returns (B, 16, 256)
        project()            : trainable linear, runs every training step
        forward()            : calls both — used only at inference time
    """

    def __init__(self, context_dim: int = 256):
        super().__init__()
        # Frozen ResNet-18 backbone up to (and including) layer3
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        # children order: conv1, bn1, relu, maxpool, layer1, layer2, layer3, layer4, avgpool, fc
        # Index 7 = layer3  →  output: (B, 256, 4, 4) for 64×64 input
        self.backbone = nn.Sequential(*list(resnet.children())[:7])
        for p in self.backbone.parameters():
            p.requires_grad = False

        # Trainable projection: 256 → context_dim
        self.proj = nn.Linear(256, context_dim)

    def _normalize(self, x):
        """Convert [-1, 1] image tensor to ImageNet-normalized tensor."""
        x_01 = (x + 1) / 2
        mean = torch.tensor([0.485, 0.456, 0.406], device=x.device).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225], device=x.device).view(1, 3, 1, 1)
        return (x_01 - mean) / std

    @torch.no_grad()
    def backbone_features(self, x):
        """Run frozen backbone only.  Call once per image during pre-computation.
        Args:  x  (B, 3, 64, 64) in [-1, 1]
        Returns:   (B, 16, 256)  — 16 spatial tokens, 256-dim each
        """
        x_norm = self._normalize(x)
        features = self.backbone(x_norm)    # (B, 256, 4, 4)
        B, C, H, W = features.shape
        tokens = features.reshape(B, C, H * W).permute(0, 2, 1)  # (B, 16, 256)
        return tokens

    def project(self, tokens):
        """Run trainable projection only.  Called every training step.
        Args:  tokens  (B, 16, 256)  — pre-computed backbone tokens
        Returns:       (B, 16, context_dim)
        """
        return self.proj(tokens)

    def forward(self, x):
        """Full forward pass (backbone + projection).  Used at inference time.
        Args:  x  (B, 3, 64, 64) in [-1, 1]
        Returns:   (B, 16, context_dim)
        """
        tokens = self.backbone_features(x)
        return self.project(tokens)

### U-Net Configuration

We use `UNet2DConditionModel` from `diffusers` with a custom configuration:

- **`in_channels=8`**: 4 channels from the noisy color latent $x_t$ + 4 channels from the grayscale latent $z_{\text{gray}}$, concatenated along the channel dimension
- **`out_channels=4`**: predicted noise $\hat{\varepsilon}$ in latent space
- **`sample_size=8`**: spatial size of the latent (8×8 for 64×64 images with 8× VAE compression)
- **`cross_attention_dim=256`**: matches our `ReferenceEncoder` output dimension
- **Block types**: We use 3 down/up blocks. Cross-attention is applied at the lower 2 spatial levels (4×4 and 2×2) where it is most effective and computationally feasible.

#### Channel flow through the U-Net

```
Input (8, 8×8)
  ├── DownBlock2D:          128ch, 8×8  — no attention (spatial guide only)
  ├── CrossAttnDownBlock2D: 256ch, 4×4  ← cross-attention with reference tokens
  ├── CrossAttnDownBlock2D: 512ch, 2×2  ← cross-attention with reference tokens
  │
  ├── Mid Block:            512ch, 2×2  (self-attn + cross-attn + ResBlocks)
  │
  ├── CrossAttnUpBlock2D:   512ch, 2×2  ← cross-attention + skip connection
  ├── CrossAttnUpBlock2D:   256ch, 4×4  ← cross-attention + skip connection
  └── UpBlock2D:            128ch, 8×8  + skip connection
Output (4, 8×8)
```

In [ ]:
# ============================================================
# U-Net: Conditional U-Net from diffusers
# ============================================================
unet = UNet2DConditionModel(
    sample_size=8,               # latent spatial size (8x8 for 64x64 images)
    in_channels=8,               # 4 (noisy latent) + 4 (grayscale latent)
    out_channels=4,              # predicted noise
    layers_per_block=2,          # 2 ResBlocks per level
    block_out_channels=(128, 256, 512),
    down_block_types=(
        "DownBlock2D",           # 128ch, 8x8  — no attention
        "CrossAttnDownBlock2D",  # 256ch, 4x4  — cross-attention with reference
        "CrossAttnDownBlock2D",  # 512ch, 2x2  — cross-attention with reference
    ),
    up_block_types=(
        "CrossAttnUpBlock2D",    # 512ch, 2x2
        "CrossAttnUpBlock2D",    # 256ch, 4x4
        "UpBlock2D",             # 128ch, 8x8
    ),
    cross_attention_dim=256,      # matches ReferenceEncoder output
).to(device)

print(f"U-Net parameters: {sum(p.numel() for p in unet.parameters()):,}")

In [ ]:
# ============================================================
# Noise Schedulers  (cosine schedule — better SNR at low timesteps)
# ============================================================
noise_scheduler = DDPMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",  # cosine schedule (was "linear")
    prediction_type="epsilon",          # predict the noise
)

ddim_scheduler = DDIMScheduler(
    num_train_timesteps=1000,
    beta_schedule="squaredcos_cap_v2",  # cosine schedule
    prediction_type="epsilon",
)

# ============================================================
# Instantiate Reference Encoder + print total trainable params
# ============================================================
ref_encoder = ReferenceEncoder(context_dim=256).to(device)

trainable_params = (
    list(unet.parameters())
    + list(ref_encoder.proj.parameters())
)
total_trainable = sum(p.numel() for p in trainable_params if p.requires_grad)
print(f"Total trainable parameters: {total_trainable:,}")

# ============================================================
# Shape test: verify everything connects correctly
# ============================================================
with torch.no_grad():
    dummy_x   = torch.randn(2, 8, 8, 8, device=device)        # (B, 8ch, 8x8)
    dummy_t   = torch.randint(0, 1000, (2,), device=device)
    dummy_ctx = torch.randn(2, 16, 256, device=device)         # (B, 16 tokens, 256)
    dummy_out = unet(dummy_x, dummy_t, encoder_hidden_states=dummy_ctx).sample
    assert dummy_out.shape == (2, 4, 8, 8), f"Unexpected shape: {dummy_out.shape}"
    print(f"Shape test passed: input {dummy_x.shape} -> output {dummy_out.shape}")

In [ ]:
# ============================================================
# EMA (Exponential Moving Average) — inspired by powerful-attention repo
#
# Maintains a smoothed copy of model weights that typically produces
# better inference quality than raw trained weights with no extra training cost.
# Decay=0.9999 is appropriate for ~100 epochs x ~1563 steps (~156K steps total).
# For shorter runs (e.g. 10 epochs), consider decay=0.999.
# ============================================================

class EMAModel:
    """Lightweight EMA wrapper around a single nn.Module.

    After each optimizer step, call ema.step(model) to update the shadow weights.
    At inference, call ema.copy_to(model) to swap in the smoothed weights,
    then ema.restore(model, backup) to restore the originals.
    """

    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.decay = decay
        # Store EMA weights as a separate state dict on CPU to save VRAM
        self.shadow = {
            name: param.data.clone().cpu()
            for name, param in model.named_parameters()
            if param.requires_grad
        }

    @torch.no_grad()
    def step(self, model: nn.Module):
        """Update EMA weights: shadow = decay * shadow + (1 - decay) * param."""
        for name, param in model.named_parameters():
            if name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(
                    param.data.cpu(), alpha=1.0 - self.decay
                )

    def copy_to(self, model: nn.Module):
        """Overwrite model parameters with the EMA shadow weights (for inference)."""
        for name, param in model.named_parameters():
            if name in self.shadow:
                param.data.copy_(self.shadow[name].to(param.device))

    def restore(self, model: nn.Module, backup: dict):
        """Restore model parameters from a backup dict."""
        for name, param in model.named_parameters():
            if name in backup:
                param.data.copy_(backup[name])


# Instantiate EMA trackers for both trainable modules
ema_unet        = EMAModel(unet,        decay=0.9999)
ema_ref_encoder = EMAModel(ref_encoder, decay=0.9999)
print("EMA trackers initialized for unet and ref_encoder.")

---
## 9. Training

### Training Loop Walkthrough

Each training step performs:

1. **Load pre-computed latents**: $z_{\text{color}} \in \mathbb{R}^{B \times 4 \times 8 \times 8}$ and $z_{\text{gray}} \in \mathbb{R}^{B \times 4 \times 8 \times 8}$
2. **Project pre-computed backbone tokens**: cached backbone features $(B, 16, 256)$ → projected context $(B, 16, 256)$ via trainable linear layer
3. **Sample noise**: $\varepsilon \sim \mathcal{N}(0, \mathbf{I})$, same shape as $z_{\text{color}}$
4. **Sample timestep**: $t \sim \text{Uniform}(0, T{-}1)$
5. **Forward diffusion**: $x_t = \sqrt{\bar{\alpha}_t}\, z_{\text{color}} + \sqrt{1{-}\bar{\alpha}_t}\, \varepsilon$
6. **Concatenate**: $\text{input} = [x_t \;\|\; z_{\text{gray}}] \in \mathbb{R}^{B \times 8 \times 8 \times 8}$
7. **Predict noise**: $\hat{\varepsilon} = \text{UNet}(\text{input},\, t,\, \texttt{encoder\_hidden\_states}{=}\text{context})$
8. **Compute loss**: $\mathcal{L} = \|\varepsilon - \hat{\varepsilon}\|^2$ (MSE)
9. **Backpropagate** and update U-Net + reference projection weights

### Optimizations

- **Pre-compute latents + backbone tokens**: We encode all images through the frozen VAE *and* the frozen ResNet-18 backbone once before training. Only the cheap trainable `proj` linear runs per step. This avoids all repeated frozen-model passes.
- **EMA weight averaging**: After each optimizer step, EMA weights are updated (decay=0.9999). EMA weights are used at inference for smoother results.
- **Mixed precision (fp16)**: The U-Net forward/backward runs in float16 via `torch.cuda.amp`, reducing VRAM usage and increasing throughput by ~1.5×.
- **Gradient clipping**: We clip gradients to `max_norm=1.0` for training stability.

In [ ]:
# ============================================================
# Pre-compute VAE latents + ResNet backbone tokens
# (avoids all repeated frozen-model passes during training)
# ============================================================
print("Pre-computing VAE latents and backbone tokens for all images...")

all_color_latents   = []
all_gray_latents    = []
all_backbone_tokens = []   # (N, 16, 256) — pre-computed ResNet features

# Use large batch for fast pre-computation (no gradients needed)
precompute_loader = DataLoader(
    dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True
)

ref_encoder.eval()

for batch in tqdm(precompute_loader, desc="Encoding"):
    color_latent    = encode_to_latent(batch["color"].to(device))               # (B, 4, 8, 8)
    gray_latent     = encode_to_latent(batch["gray"].to(device))                # (B, 4, 8, 8)
    backbone_tokens = ref_encoder.backbone_features(batch["ref"].to(device))    # (B, 16, 256)

    all_color_latents.append(color_latent.cpu())
    all_gray_latents.append(gray_latent.cpu())
    all_backbone_tokens.append(backbone_tokens.cpu())

all_color_latents   = torch.cat(all_color_latents,   dim=0)
all_gray_latents    = torch.cat(all_gray_latents,    dim=0)
all_backbone_tokens = torch.cat(all_backbone_tokens, dim=0)

print(f"Color latents:   {all_color_latents.shape}")    # (N, 4, 8, 8)
print(f"Gray latents:    {all_gray_latents.shape}")     # (N, 4, 8, 8)
print(f"Backbone tokens: {all_backbone_tokens.shape}")  # (N, 16, 256)

# Note: backbone tokens are fixed at pre-computation time.
# The same reference is used for each sample across all epochs.
# This matches the original notebook's behavior and is acceptable for simplicity.

# Create a new dataset from pre-computed latents + backbone tokens
class LatentDataset(Dataset):
    def __init__(self, color_latents, gray_latents, backbone_tokens):
        self.color_latents   = color_latents
        self.gray_latents    = gray_latents
        self.backbone_tokens = backbone_tokens

    def __len__(self):
        return len(self.color_latents)

    def __getitem__(self, idx):
        return (
            self.color_latents[idx],
            self.gray_latents[idx],
            self.backbone_tokens[idx],   # (16, 256) — NOT a raw pixel image
        )

latent_dataset = LatentDataset(all_color_latents, all_gray_latents, all_backbone_tokens)
latent_loader  = DataLoader(
    latent_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True
)
print(f"\nLatent dataloader ready: {len(latent_loader)} batches/epoch")

In [ ]:
# ============================================================
# Training Loop
# ============================================================
NUM_EPOCHS = 100
LR         = 2e-4
GRAD_CLIP  = 1.0

optimizer    = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
scaler       = GradScaler()

# Fixed test sample for periodic visualization
test_color_latent    = all_color_latents[0:1].to(device)
test_gray_latent     = all_gray_latents[0:1].to(device)
test_backbone_tokens = all_backbone_tokens[0:1].to(device)   # (1, 16, 256)

losses = []
unet.train()
ref_encoder.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss  = 0.0
    num_batches = 0

    for color_latent, gray_latent, backbone_tokens in latent_loader:
        color_latent    = color_latent.to(device)
        gray_latent     = gray_latent.to(device)
        backbone_tokens = backbone_tokens.to(device)   # (B, 16, 256)
        B = color_latent.shape[0]

        # 1. Project pre-computed backbone tokens → context for cross-attention
        #    Only the trainable linear proj runs here; backbone is frozen + pre-computed
        context = ref_encoder.proj(backbone_tokens)    # (B, 16, 256)

        # 2. Sample noise and random timesteps
        noise     = torch.randn_like(color_latent)
        timesteps = torch.randint(
            0, noise_scheduler.config.num_train_timesteps, (B,), device=device
        ).long()

        # 3. Forward diffusion: add noise to color latent
        noisy_latent = noise_scheduler.add_noise(color_latent, noise, timesteps)

        # 4. Concatenate noisy color latent with grayscale latent  (B, 8, 8, 8)
        model_input = torch.cat([noisy_latent, gray_latent], dim=1)

        # 5. Predict noise (mixed precision)
        optimizer.zero_grad()
        with autocast():
            noise_pred = unet(model_input, timesteps, encoder_hidden_states=context).sample
            loss       = F.mse_loss(noise_pred, noise)

        # 6. Backprop with gradient scaling
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(trainable_params, GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()

        # 7. Update EMA weights for both trainable modules
        ema_unet.step(unet)
        ema_ref_encoder.step(ref_encoder)

        epoch_loss  += loss.item()
        num_batches += 1

    lr_scheduler.step()
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} | LR: {lr_scheduler.get_last_lr()[0]:.2e}")

    # Quick DDIM visualization every 20 epochs — use EMA weights for preview
    if (epoch + 1) % 20 == 0:
        unet.eval()
        ref_encoder.eval()
        with torch.no_grad():
            # Temporarily swap in EMA weights for the preview
            unet_backup = {n: p.data.clone() for n, p in unet.named_parameters() if n in ema_unet.shadow}
            ref_backup  = {n: p.data.clone() for n, p in ref_encoder.named_parameters() if n in ema_ref_encoder.shadow}
            ema_unet.copy_to(unet)
            ema_ref_encoder.copy_to(ref_encoder)

            ctx    = ref_encoder.proj(test_backbone_tokens)   # (1, 16, 256)
            latent = torch.randn(1, 4, 8, 8, device=device)
            ddim_scheduler.set_timesteps(50)
            for t in ddim_scheduler.timesteps:
                inp  = torch.cat([latent, test_gray_latent], dim=1)
                pred = unet(inp, t, encoder_hidden_states=ctx).sample
                latent = ddim_scheduler.step(pred, t, latent).prev_sample
            preview = decode_from_latent(latent)

            # Restore original weights
            ema_unet.restore(unet, unet_backup)
            ema_ref_encoder.restore(ref_encoder, ref_backup)

        fig, axes = plt.subplots(1, 3, figsize=(9, 3))
        gt = decode_from_latent(test_color_latent)
        for ax, img, title in zip(axes,
            [decode_from_latent(test_gray_latent), preview, gt],
            ["Grayscale", f"Colorized EMA (ep {epoch+1})", "Ground Truth"]):
            ax.imshow((img[0].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1))
            ax.set_title(title)
            ax.axis("off")
        plt.tight_layout()
        plt.show()
        unet.train()
        ref_encoder.train()

    # Checkpoint every 25 epochs — save both raw and EMA state dicts
    if (epoch + 1) % 25 == 0:
        torch.save({
            "epoch":           epoch,
            "unet":            unet.state_dict(),
            "ref_encoder":     ref_encoder.state_dict(),
            "ema_unet_shadow": ema_unet.shadow,
            "ema_ref_shadow":  ema_ref_encoder.shadow,
            "optimizer":       optimizer.state_dict(),
            "losses":          losses,
        }, f"/content/checkpoint_epoch{epoch+1}.pt")
        print(f"  Checkpoint saved.")

print("\nTraining complete!")

In [ ]:
# ============================================================
# Plot training loss curve
# ============================================================
plt.figure(figsize=(10, 4))
plt.plot(losses, linewidth=1.5)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Inference with DDIM

Now that the model is trained, we sample colorized images using **DDIM** (Denoising Diffusion Implicit Models):

1. **Start from pure noise**: $z_T \sim \mathcal{N}(0, \mathbf{I})$ in latent space $(1, 4, 8, 8)$
2. **Set timestep schedule**: 50 evenly spaced steps from $T{-}1$ down to $0$
3. **At each step**: concatenate with $z_{\text{gray}}$, predict noise, take DDIM step
4. **Decode**: pass the final denoised latent through the VAE decoder

The `DDIMScheduler.step()` method implements the DDIM update rule we saw in Section 2:
- First, it predicts $\hat{x}_0$ from the current $x_t$ and the predicted noise
- Then, it computes $x_{t-1}$ as a deterministic interpolation toward $\hat{x}_0$

Because DDIM is deterministic ($\eta = 0$), the same noise seed always produces the same output. This makes results reproducible and enables interpolation in the noise space.

In [ ]:
# ============================================================
# DDIM Sampling Function  (uses EMA weights for inference quality)
# ============================================================

@torch.no_grad()
def sample_ddim(gray_image, ref_image, num_steps=50, seed=None):
    """Generate a colorized image from a grayscale input and a color reference.

    Args:
        gray_image: (1, 3, 64, 64) grayscale image in [-1, 1] (3-channel)
        ref_image:  (1, 3, 64, 64) reference color image in [-1, 1]
        num_steps:  number of DDIM denoising steps
        seed:       optional random seed for reproducibility
    Returns:
        colorized: (1, 3, 64, 64) in [-1, 1]
    """
    unet.eval()
    ref_encoder.eval()

    # Temporarily swap in EMA weights for inference quality
    unet_backup = {n: p.data.clone() for n, p in unet.named_parameters() if n in ema_unet.shadow}
    ref_backup  = {n: p.data.clone() for n, p in ref_encoder.named_parameters() if n in ema_ref_encoder.shadow}
    ema_unet.copy_to(unet)
    ema_ref_encoder.copy_to(ref_encoder)

    try:
        # Encode inputs
        gray_latent = encode_to_latent(gray_image.to(device))   # (1, 4, 8, 8)
        # Full forward pass: backbone + proj (ref_image is raw pixels here)
        context = ref_encoder(ref_image.to(device))             # (1, 16, 256)

        # Start from pure noise
        if seed is not None:
            generator = torch.Generator(device=device).manual_seed(seed)
        else:
            generator = None
        latent = torch.randn(1, 4, 8, 8, device=device, generator=generator)

        # DDIM denoising loop
        ddim_scheduler.set_timesteps(num_steps)
        for t in ddim_scheduler.timesteps:
            model_input = torch.cat([latent, gray_latent], dim=1)  # (1, 8, 8, 8)
            noise_pred  = unet(model_input, t, encoder_hidden_states=context).sample
            latent      = ddim_scheduler.step(noise_pred, t, latent).prev_sample

        # Decode to pixel space
        colorized = decode_from_latent(latent)
    finally:
        # Always restore original (non-EMA) weights after inference
        ema_unet.restore(unet, unet_backup)
        ema_ref_encoder.restore(ref_encoder, ref_backup)

    return colorized

In [ ]:
# ============================================================
# Run inference on test samples
# ============================================================
NUM_TEST = 6

# Sample random indices from the dataset
test_indices = random.sample(range(len(dataset)), NUM_TEST)

fig, axes = plt.subplots(NUM_TEST, 4, figsize=(14, 3.5 * NUM_TEST))
col_titles = ["Grayscale (input)", "Reference (condition)", "Colorized (output)", "Ground Truth"]

for row, idx in enumerate(test_indices):
    sample = dataset[idx]
    gray = sample["gray"].unsqueeze(0)    # (1, 3, 64, 64)
    ref = sample["ref"].unsqueeze(0)      # (1, 3, 64, 64)
    color = sample["color"].unsqueeze(0)  # (1, 3, 64, 64)

    colorized = sample_ddim(gray, ref, num_steps=50, seed=42)

    images = [gray, ref, colorized.cpu(), color]
    for col, (img, title) in enumerate(zip(images, col_titles)):
        arr = img[0].permute(1, 2, 0).numpy() * 0.5 + 0.5
        axes[row, col].imshow(arr.clip(0, 1))
        if row == 0:
            axes[row, col].set_title(title, fontsize=12)
        axes[row, col].axis("off")

plt.suptitle("Conditional Diffusion Colorization Results", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Ablation: Same grayscale, different references
# This proves the model uses cross-attention conditioning!
# ============================================================
NUM_REFS = 4

# Pick one grayscale image
base_sample = dataset[0]
gray = base_sample["gray"].unsqueeze(0)

# Pick references from different categories
ref_indices = random.sample(range(len(dataset)), NUM_REFS)

fig, axes = plt.subplots(1, NUM_REFS + 1, figsize=(4 * (NUM_REFS + 1), 4))

# Show the grayscale input
axes[0].imshow((gray[0].permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1))
axes[0].set_title("Grayscale input", fontsize=11)
axes[0].axis("off")

for i, ref_idx in enumerate(ref_indices):
    ref = dataset[ref_idx]["color"].unsqueeze(0)  # use color as reference
    colorized = sample_ddim(gray, ref, num_steps=50, seed=42)

    # Show colorized output with the reference as a small inset
    arr = colorized[0].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5
    axes[i + 1].imshow(arr.clip(0, 1))

    # Add reference image as inset
    ref_arr = ref[0].permute(1, 2, 0).numpy() * 0.5 + 0.5
    inset = axes[i + 1].inset_axes([0.65, 0.65, 0.33, 0.33])
    inset.imshow(ref_arr.clip(0, 1))
    inset.axis("off")
    inset.set_title("ref", fontsize=8)

    axes[i + 1].set_title(f"Reference {i+1}", fontsize=11)
    axes[i + 1].axis("off")

plt.suptitle("Same grayscale + different references → different colorizations", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Evaluation: Histogram Intersection Score (HIS) + LPIPS
#
# We evaluate over a held-out subset of the CIFAR-10 test split.
#
# HIS (Histogram Intersection Score):
#   Measures how similar the AB color distribution of the colorized
#   image is to the ground truth. Computed in CIE-LAB space.
#   Score in [0, 1]; higher is better.
#
# LPIPS (Learned Perceptual Image Patch Similarity):
#   Perceptual distance using deep AlexNet features. Lower is better.
#
# Inspired by the evaluation suite in the powerful-attention repo.
# ============================================================
import cv2

# Build a small test dataset from CIFAR-10 test split
cifar_test = CIFAR10(root="/content/cifar10", train=False, download=True)
test_category = {}
for img_array, label in zip(cifar_test.data, cifar_test.targets):
    test_category.setdefault(label, []).append(Image.fromarray(img_array))
test_dataset = CIFARColorizationDataset(test_category, image_size=64)
print(f"Test dataset size: {len(test_dataset)} images")

# Initialize LPIPS metric
lpips_fn = lpips.LPIPS(net='alex').to(device)
lpips_fn.eval()


def compute_his(gt_rgb: np.ndarray, pred_rgb: np.ndarray, bins: int = 32) -> float:
    """Histogram Intersection Score on A+B channels of CIE-LAB.

    Args:
        gt_rgb:   (H, W, 3) uint8 RGB numpy array — ground truth
        pred_rgb: (H, W, 3) uint8 RGB numpy array — colorized prediction
        bins:     number of histogram bins per channel
    Returns:
        float in [0, 1]
    """
    # OpenCV uses BGR order
    lab_gt   = cv2.cvtColor(gt_rgb[:, :, ::-1],   cv2.COLOR_BGR2LAB)
    lab_pred = cv2.cvtColor(pred_rgb[:, :, ::-1],  cv2.COLOR_BGR2LAB)
    scores = []
    for ch in [1, 2]:  # A and B channels only
        h_gt   = np.histogram(lab_gt[:, :, ch],   bins=bins, range=(0, 256), density=True)[0]
        h_pred = np.histogram(lab_pred[:, :, ch], bins=bins, range=(0, 256), density=True)[0]
        # Intersection; multiply by bin width (256/bins) to normalize to [0, 1]
        scores.append(np.minimum(h_gt, h_pred).sum() * (256 / bins))
    return float(np.mean(scores))


def to_uint8(tensor):
    """Convert (1, 3, H, W) tensor in [-1,1] to (H, W, 3) uint8 numpy array."""
    arr = (tensor[0].cpu().permute(1, 2, 0).numpy() * 0.5 + 0.5).clip(0, 1)
    return (arr * 255).astype(np.uint8)


NUM_EVAL = 200   # evaluate on 200 random samples (full 10k is slow at inference time)
eval_indices = random.sample(range(len(test_dataset)), NUM_EVAL)

his_scores   = []
lpips_scores = []

print(f"Evaluating on {NUM_EVAL} samples...")
for idx in tqdm(eval_indices):
    sample    = test_dataset[idx]
    gray      = sample["gray"].unsqueeze(0)    # (1, 3, 64, 64)
    ref       = sample["ref"].unsqueeze(0)     # (1, 3, 64, 64)
    color     = sample["color"].unsqueeze(0)   # (1, 3, 64, 64)  ground truth

    colorized = sample_ddim(gray, ref, num_steps=50, seed=0)   # (1, 3, 64, 64)

    # HIS
    his_scores.append(compute_his(to_uint8(color), to_uint8(colorized)))

    # LPIPS (expects tensors in [-1, 1])
    with torch.no_grad():
        lpips_scores.append(
            lpips_fn(colorized.to(device), color.to(device)).item()
        )

print(f"\n--- Evaluation Results (n={NUM_EVAL}) ---")
print(f"HIS   (AB histogram intersection, higher is better): {np.mean(his_scores):.4f} +/- {np.std(his_scores):.4f}")
print(f"LPIPS (perceptual distance,        lower is better):  {np.mean(lpips_scores):.4f} +/- {np.std(lpips_scores):.4f}")

---
## 11. Discussion

### What We Built

We built a **conditional latent diffusion model** for image colorization that uses the same conditioning mechanism as Stable Diffusion:

- **Frozen VAE**: compresses 64×64 images into 8×8×4 latents, enabling efficient diffusion
- **Channel concatenation**: provides the grayscale structural guide, maintaining spatial alignment
- **Cross-attention conditioning**: injects the reference image's color palette via `encoder_hidden_states`, allowing the U-Net to selectively attend to relevant color regions

### Connection to Stable Diffusion

Our pipeline is a faithful simplification of the Stable Diffusion architecture. The key substitution:

| Stable Diffusion | Our Model |
|---|---|
| CLIP text encoder → (B, 77, 768) | ResNet-18 image encoder → (B, 16, 256) |
| Text prompt describes desired content | Reference image provides desired colors |
| Generates from text | Colorizes from grayscale + reference |
| Raw trained weights | **EMA-smoothed weights** for inference quality |

The mechanism is identical: condition tokens enter through `encoder_hidden_states` and interact with U-Net features via cross-attention. The model learns *what* to attend to in the condition (color information) and *where* to apply it (guided by the grayscale structure).

### What We Improved (vs. Baseline)

- **CIFAR-10 dataset**: Replaced COCO (URL-scraped, slow) with CIFAR-10 (built-in torchvision, 50K images, zero download friction)
- **Pre-computed backbone tokens**: The frozen ResNet-18 backbone now runs once at pre-computation time; only the cheap linear `proj` layer runs per training step
- **Cosine noise schedule**: `squaredcos_cap_v2` gives better signal-to-noise at low timesteps compared to the linear schedule
- **EMA weights**: `EMAModel` maintains a smoothed copy of U-Net + ref encoder weights (decay=0.9999); used at inference for higher quality outputs
- **Evaluation metrics**: HIS (histogram intersection in LAB space) and LPIPS (perceptual similarity) quantify colorization quality

### Potential Failure Modes

- **Color bleeding**: colors from the reference may spread to incorrect regions
- **Reference ignored**: the model may learn to produce "average" colors, underusing the reference
- **Grayscale passthrough**: the model may output a slightly tinted grayscale instead of rich colors

These can be addressed with more training data, higher capacity, multi-level cross-attention, or classifier-free guidance.

## 12. Extension Exercises

Here are several directions to deepen your understanding:

1. **CLIP image encoder**: Replace the ResNet-18 reference encoder with [CLIP ViT](https://huggingface.co/openai/clip-vit-base-patch32). Does semantic understanding of the reference image improve colorization quality?

2. **Classifier-free guidance**: During training, randomly replace the reference context with zeros 10% of the time (`context = torch.zeros_like(context)`). At inference, blend conditional and unconditional predictions: $\hat{\varepsilon} = \varepsilon_{\text{uncond}} + s \cdot (\varepsilon_{\text{cond}} - \varepsilon_{\text{uncond}})$, where $s > 1$ amplifies the effect of conditioning.

3. **Higher resolution**: Increase from 128×128 to 256×256. The latent space becomes 32×32×4. How does this affect training time and quality?

4. **Perceptual loss**: Add an LPIPS perceptual loss alongside MSE. Does this improve the visual quality of colorized images?

5. **Quantitative evaluation**: Compute FID (Fréchet Inception Distance) or LPIPS between colorized outputs and ground truth. How does performance vary with the number of DDIM steps (10, 25, 50, 100)?